例:
```bash
f=../data/sample_151507_anno.h5
for i in {1..5}
    do
        python -u run_DSSC.py --n_clusters -1 --data_file $f --save_dir out_151507 \
            --final_labels pred.csv --final_latent_file latent.csv --run $i \
            --ml_file ../example_constraints/sample_151507_mlFromMarks_test.txt --cl_file ../example_constraints/sample_151507_clFromMarks_test.txt
    done
```

# 断点 #1

src/run_DSSC.py:147，model.train_model(...) 被调用的那一刻 —— 三个输入文件都已完成加载/变换，正要第一次进入模型。

## 1. f=../data/sample_151507_anno.h5

### 读取模块：run_DSSC.py:53-62

| h5 字段 | 原始形状/类型	   | 说明                     |
| ------ | ---------------- | ------------------------ | 
| X	| (4226, 24402) float64 | 原始 counts 矩阵 | 
| Pos | (2, 4226) int32 | 空间坐标（2 行） | 
| Y | (4226,) bytes | 金标准层标签 Layer1..6/WM/NA | 
| Gene | (24402,) | 注意：当前代码没有读取 Gene（只读 X/Pos/Y） | 

In [17]:
import h5py
import numpy as np

data_mat = h5py.File("../../data/sample_151507_anno.h5")
x = np.array(data_mat['X'])
pos = np.array(data_mat['Pos'])
if pos.shape[1] != 2: 
    pos = pos.T
y = np.array(data_mat['Y'])
data_mat.close()

print(x.shape)
print(pos.shape)
print(y.shape)

(4226, 24402)
(4226, 2)
(4226,)


/tmp/ipykernel_591778/4125188590.py:4: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  data_mat = h5py.File("../../data/sample_151507_anno.h5")


In [12]:
print(x, '\n')
print(pos, '\n')
print(y)

[[ 0.  0.  0. ...  0.  5.  0.]
 [ 0.  0.  0. ...  0. 50.  0.]
 [ 0.  0.  0. ...  0. 22.  0.]
 ...
 [ 0.  0.  0. ...  0.  7.  0.]
 [ 0.  0.  0. ...  1. 26.  0.]
 [ 0.  0.  0. ...  0. 31.  0.]] 

[[  0  16]
 [ 50 102]
 [  3  43]
 ...
 [ 60  30]
 [ 45  27]
 [  7  51]] 

[b'Layer1' b'Layer3' b'Layer1' ... b'WM' b'Layer6' b'Layer1']


### 变换过程：
1. run_DSSC.py:56-57：因为 Pos.shape[1] != 2，执行 pos = pos.T → (4226, 2)。（这正是 README 里对 osmFISH 修复的同一处逻辑。）
2. run_DSSC.py:65-69：f = where(Y != "NA")，剔除 5 个 NA 细胞 → x=(4221,24402)、pos=(4221,2)、y=(4221,)。注意：这里重排了细胞索引，而 ml/cl 文件也是按同一顺序、在剔 NA 后生成的，所以索引仍能对齐。
3. run_DSSC.py:72-75：n_clusters = len(unique(y)) = 7（Layer1–6 + WM）。Y 到此只用于确定簇数。
4. 构造 AnnData（run_DSSC.py:87-92）：obs 写入 cell_labels、batch=1、pos_x/pos_y。
5. preprocessing_rna（preprocess.py:78-151）：
- 丢弃 ERCC/MT-/mt- 基因（preprocess.py:118）；
- count_X = adata.X（保留原始 counts 副本，preprocess.py:124）；
- normalize_per_cell（preprocess.py:133）→ log1p（preprocess.py:137）→ 选 2000 个高变基因并 subset（preprocess.py:141）；
- 返回 adata=(4221,2000)（log 归一化 HVG）和 count_X=(4221,2000)（原始 counts，仅 HVG 列）。
- size_factors = n_counts（归一化前的每细胞总 counts，preprocess.py:134）。
6. run_DSSC.py:94-99：pos_=(4221,2) 从 obs 取回；y 重编码为 cat.codes 0–6。
进入模型的分流（都在 model.train_model，DSSC.py:148）：

In [ ]:
import pandas as pd
import scanpy as sc

print(f'需剔除细胞数: {(y.astype(np.str) == "NA").sum()}') # 需剔除的细胞数：5
# 2 剔除 NA
f = np.where(y.astype(np.str) != "NA")[0]
y = y[f]
x = x[f,:]
pos = pos[f,:]
pos = pos.astype(np.float)
# 3 获取簇数
n_clusters = np.shape(np.unique(y))[0] # 共有几簇？
print(f'簇数: {n_clusters}')
# 4 构建 Anndata 对象
obs = pd.DataFrame()
obs['cell_labels'] = pd.Categorical(y, categories=np.unique(y), ordered=False)
obs['batch'] = pd.Categorical(np.repeat(1, x.shape[0]), categories=[1], ordered=False)
obs['pos_x'] = pos[:,0]
obs['pos_y'] = pos[:,1]
adata = sc.AnnData(X=x, obs=obs)

需剔除细胞数: 5
簇数: 7


(7,)